# Обучение RNN

In [2]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader


# do not change the code in the block below
# __________start of block__________
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('{} device is available'.format(device))
# __________end of block__________

# Для воспроизводимости результатов фиксируем seed
seed = 33

if torch.cuda.is_available() and device.type == 'cuda':
    np.random.seed(seed)
     # Если используете CUDA, устанавливаем seed для GPU                
    torch.cuda.manual_seed_all(seed)
    # Для полной воспроизводимости также можно зафиксировать алгоритмы cuDNN
    torch.backends.cudnn.deterministic = True   
    torch.backends.cudnn.benchmark = False
else:
    # Устанавливаем seed для CPU генератора случайных чисел PyTorch
    np.random.seed(seed)
    torch.manual_seed(seed)


# Несколько оптимизаций которые могут ускорить процесс обучения
if torch.backends.cudnn.is_available():
    print(f'Using cudnn version: {torch.backends.cudnn.version()}')
    if device.type == 'cuda':
        torch.backends.cudnn.enabled = True
        torch.backends.cudnn.benchmark = True

cuda device is available
Using cudnn version: 90100


## Загрузка данных

In [3]:
class OneginDataset(Dataset):
    """Face Landmarks dataset."""

    def __init__(self, path, seq_length, transform=None):
        # Откроем файл и считаем все строки текста
        with open('onegin.txt', 'r') as iofile:
            self.text = iofile.readlines()
    
        self.text = "".join([x.replace('\t\t', '').lower() for x in self.text])


        # Создадим словари для отображения символов в токены и обратно
        self.tokens = sorted(set(self.text)) + ['<sos>']
        self.num_tokens = len(self.tokens)
        self.token_to_idx = {x: idx for idx, x in enumerate(self.tokens)}
        self.idx_to_token = {idx: x for idx, x in enumerate(self.tokens)}

        # Преобразуем текст в последовательность токенов
        self.text_encoded = torch.tensor([self.token_to_idx[x] for x in self.text])
        self.text_encoded = torch.split(self.text_encoded, seq_length)
        self.text_encoded = tuple(torch.cat((t, torch.ones(seq_length - t.size(0), dtype=t.dtype))) if t.size(0) < seq_length else t for t in self.text_encoded) # Говно паддинг
        self.text_encoded = tuple(torch.cat((torch.tensor([83]), t)) for t in self.text_encoded)


    def __len__(self):
        return len(self.text_encoded)

    def __getitem__(self, idx):
        return self.text_encoded[idx]

Проверка датасета и загрузчика данных

In [4]:
onegin_dataset = OneginDataset('onegin.txt', seq_length=100)

dataloader = DataLoader(onegin_dataset, batch_size=256, shuffle=True, num_workers=4)

## Определение модели

In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Variable

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, n_layers=1):
        super(RNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        
        self.encoder = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, n_layers)
        self.decoder = nn.Linear(hidden_size, output_size)
    
    def forward(self, input, hidden):
        input = self.encoder(input.view(1, -1))
        output, hidden = self.gru(input.view(1, 1, -1), hidden)
        output = self.decoder(output.view(1, -1))
        return output, hidden

    def init_hidden(self):
        return Variable(torch.zeros(self.n_layers, 1, self.hidden_size))

Функция обучения нейронной сети:

In [ ]:
def train(model, dataloader, num_epochs, learning_rate, criterion, optimizer):
    
    model.to(device)
    model.train()

    # Итерируемся по эпохам
    for epoch in range(num_epochs):
        
        total_loss = 0
                
        # Итерируемся по батчам
        for batch in dataloader:
            batch = batch.to(device)

            optimizer.zero_grad()

            hidden = model.init_hidden().to(device)
            model.zero_grad()
            loss = 0

            # Для каждого примера в батче
            for example in batch:

                input = example[:-1]
                target = example[1:]

                # Для каждого символа в примере
                for chr_num in range(len(input)):
                    output, hidden = model(input[chr_num], hidden)
                    loss += criterion(output, target[chr_num].unsqueeze(0))
                    
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}')
            

model = RNN(input_size=onegin_dataset.num_tokens, hidden_size=128, output_size=onegin_dataset.num_tokens, n_layers=2)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train(model, dataloader, num_epochs=10, criterion=criterion, optimizer=optimizer)

Epoch 1/10, Loss: 100967.701171875
Epoch 2/10, Loss: 85966.50260416667
Epoch 3/10, Loss: 79424.95963541667
Epoch 4/10, Loss: 78271.18619791667
Epoch 5/10, Loss: 77310.66471354167
Epoch 6/10, Loss: 75841.29427083333
Epoch 7/10, Loss: 74105.45768229167
Epoch 8/10, Loss: 72068.12955729167
Epoch 9/10, Loss: 69885.34765625
Epoch 10/10, Loss: 68014.13802083333


In [21]:
def train(model, dataloader, num_epochs, criterion, optimizer):
    
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
                
        for batch in dataloader:
            batch = batch.to(device)
            optimizer.zero_grad()
            hidden = model.init_hidden().to(device)
            loss = 0

            for example in batch:
                input = example[:-1]
                target = example[1:]

                for chr_num in range(len(input)):
                    output, hidden = model(input[chr_num], hidden)
                    loss += criterion(output, target[chr_num].unsqueeze(0))
                    
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}')
            

learning_rate=0.001

model = RNN(input_size=onegin_dataset.num_tokens, hidden_size=128, output_size=onegin_dataset.num_tokens, n_layers=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train(model, dataloader, num_epochs=10, criterion=criterion, optimizer=optimizer)

Epoch 1/10, Loss: 101870.20572916667
Epoch 2/10, Loss: 87989.33072916667


KeyboardInterrupt: 

## Вариант 2

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)
        
        self.hidden_size = hidden_size

    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
        output, h = self.rnn(x.unsqueeze(2), h0)
        return self.output(output)


In [ ]:
loss_history = []
def train(model, criterion, optimizer, batch_generator, steps):
    model.train()
    for step in tqdma(range(steps)):
        batch = next(batch_generator())
        input_data = torch.tensor(batch[:, :-1], dtype=torch.float32) # except last token
        target = torch.tensor(batch[:, 1:]) # except first token

        output = model(input_data)
        
        loss = criterion(output.transpose(1,2), target)
        
        optimizer.zero_grad()
        
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        if step % 100 == 0 or step == steps - 1:
            print(
                f"Step {step}, Loss: {np.mean(loss_history[-100:])}"
            )


In [ ]:
n_hidden = 512
n_input = 1
model = RNN(n_input, n_hidden, len(token_to_idx)).to(device)
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
train(model, criterion, optimizer, generate_chunk, 500)